In [ ]:
# import os
# import sys
# project_path = os.path.abspath(os.path.join(os.getcwd(), "../../"))
# sys.path.append(project_path)
#
# requirements_path = os.path.join(project_path, "SECONDARY/requirements.txt")
# !{sys.executable} -m pip install -r "{requirements_path}"

In [ ]:
import os
import sys
import time
!{sys.executable} --version
if sys.version_info.minor == 8:
    raise RuntimeError('USE JUPYTER KERNEL VENV 3.10/310/DEFAULT INSTEAD')

!cd /workspace/CRYPTO_MACAQUES && pip install .
!cd /home/crypto/CRYPTO_MACAQUES && python3 -m venv venv && . venv/bin/activate && !{sys.executable} -m pip install .
!cd $HOME/Documents/GitHub/CRYPTO_MACAQUES && python3 -m venv venv && . venv/bin/activate && !{sys.executable} -m pip install .
!cd $HOME/Documents/GitHub/CRYPTO_MACAQUES && python3 -m venv venv && source venv/bin/activate && !{sys.executable} -m pip install .

from IPython.core.display_functions import clear_output
clear_output()

In [ ]:
%load_ext autoreload
%autoreload

from datetime import datetime
from datetime import timedelta
from dateutil import parser
from SRC.LIBRARIES.new_data_utils import fetch
from SRC.CORE._CONSTANTS import _KIEV_TIMESTAMP

sym='ZEC'
symbol = f'{sym}USDT'
discretization = '1H'
display_start_date_str = '2026-01-01 00:00' # дата, с которой вы хотите отображать '2026-03-25 14:30:00' -- формат даты и времени
load_end_date = datetime.now()# parser.parse("2025-12-31")
measure_percentile = 0.40 # 0.5 для старого поведения
use_candle_size_instead_of_shadow=True
use_mrc=True
use_mrc_r2=True
use_mrc_s2=True
use_stop_loss=True
capital_per_trade = 1000  # выделенный капитал на сделку
commission_rate = 0.00075
base_level = -0.618
# position_weights = [0.2, 0.3, 0.5]#[0.1, 0.2, 0.3, 0.4]#[0.07, 0.12, 0.20, 0.28, 0.33]#
position_weights = [0.1, 0.2, 0.3, 0.4]
# position_weights = [0.15, 0.15, 0.25, 0.45]
# position_weights = [0.07, 0.12, 0.20, 0.28, 0.33]

if abs(sum(position_weights) - 1.0) > 1e-9:
    os.system('say "ERROR! ERROR! ERROR! Check position weights"')  # голосовое оповещение
    raise ValueError(f"position_weights sum {sum(position_weights)} is not equal 1")

levels = [base_level - (i-1)*1.0 for i in range(1, len(position_weights)+1)]   # [-0.618, -1.618, -2.618, ...]
sl_level = levels[-1] - 1.0   # стоп-лосс на 1.0 ниже последнего уровня
position_sizes = {}

for i, level in enumerate(levels):
    position_sizes[level] = capital_per_trade * position_weights[i]

market_type = 'SPOT'
mrc_buffer = 1000
rsi_buffer = 200
stoch_buffer = 17
macd_buffer = 200
atr_buffer = 200
display_start_dt = parser.parse(display_start_date_str)
start_time = time.perf_counter()

# Определяем таймфрейм в секундах
discretization_value = int(discretization[:-1])
timeframe_seconds = {
    'M': discretization_value * 60,
    'H': discretization_value * 3600,
    'D': discretization_value * 86400
}.get(discretization[-1], 1800)

# Рассчитываем дату начала загрузки с буфером
load_start_dt = display_start_dt - timedelta(seconds=timeframe_seconds * mrc_buffer)

mrc_df = fetch(market_type, symbol, discretization, load_start_dt, load_end_date)
df = mrc_df.iloc[mrc_buffer:].copy()
df_1m = df if discretization == "1M" else fetch(market_type, symbol, "1M", display_start_dt, load_end_date)

In [ ]:
def calculate_current_level_profit_usd(size, price_end, price_start, commission_rate):
    """
    Точный расчет прибыли для одной позиции с учетом комиссий.
    Использует последовательное вычисление: покупка, затем продажа.
    """
    # Покупка: списываем комиссию с депозита
    amount_invested = size
    commission_buy = amount_invested * commission_rate
    net_invested = amount_invested - commission_buy
    quantity = net_invested / price_start

    # Продажа: выручка и комиссия
    gross_revenue = quantity * price_end
    commission_sell = gross_revenue * commission_rate
    net_revenue = gross_revenue - commission_sell

    profit = net_revenue - amount_invested
    commission_total = commission_buy + commission_sell
    return profit, commission_total

def calculate_stop_loss_level_loss_usd(level_prices, sl_level, col, commission_rate):
    """
    Убыток по позиции, открытой на уровне col и закрытой по стоп-лоссу sl_level.
    """
    price_start = level_prices[col]
    price_end = level_prices[sl_level]
    size = position_sizes[col]
    # Покупка
    commission_buy = size * commission_rate
    net_invested = size - commission_buy
    quantity = net_invested / price_start
    # Продажа
    gross_revenue = quantity * price_end
    commission_sell = gross_revenue * commission_rate
    net_revenue = gross_revenue - commission_sell
    loss = net_revenue - size  # отрицательное число
    return loss

def calculate_stop_loss_level_commission_usd(level_prices, sl_level, col, commission_rate):
    """
    Комиссия по позиции, закрытой по стоп-лоссу.
    """
    price_start = level_prices[col]
    size = position_sizes[col]
    commission_buy = size * commission_rate
    quantity = (size - commission_buy) / price_start
    price_end = level_prices[sl_level]
    commission_sell = (quantity * price_end) * commission_rate
    return commission_buy + commission_sell

def format_duration(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = seconds % 60

    if hours > 0:
        return f"{hours} h {minutes} min {secs:.0f} sec"
    elif minutes > 0:
        return f"{minutes} min {secs:.0f} sec"
    else:
        return f"{secs:.0f} sec"

def analyze_target_candle_outcome(df_trading, df_1m, target_idx, trading_timeframe):
    import math

    target_data = df_trading.loc[[target_idx]]
    high = target_data['high'].iloc[0]
    low = target_data['low'].iloc[0]

    # логарифмическая шкала от минимума к максимуму
    log_start = math.log(low)
    log_end = math.log(high)
    log_range = log_end - log_start

    all_levels = [0] + levels + [sl_level]
    level_prices = {level: math.exp(log_start + log_range * level) for level in all_levels}

    # Определяем начало анализа после закрытия ЦС
    if trading_timeframe == '1D':
        start_analysis = target_idx + pd.Timedelta(days=1)
    else:
        interval_minutes = {'1M':1,'5M':5,'15M':15,'30M':30,'1H':60,'4H':240,'8H':480}.get(trading_timeframe,1)
        start_analysis = target_idx + pd.Timedelta(minutes=interval_minutes)

    df_1m_after = df_1m.loc[start_analysis:].copy()
    if len(df_1m_after) == 0:
        return {'status':'in_progress','level':None,'timestamp':None,
                'fill_start_level':None,'fill_end_level':None,
                'profit_usd':0,'profit_pct':0,'level_return_pct':0,
                'commission_usd':0,'details':'No data after TC'}

    current_deep_level = None
    level_index = -1

    for idx, row in df_1m_after.iterrows():
        current_high = row['high']
        current_low = row['low']

        # Проверяем достижение нового глубокого уровня
        for i, lvl in enumerate(levels):
            if current_low <= level_prices[lvl] <= current_high:
                if current_deep_level is None or lvl < current_deep_level:
                    current_deep_level = lvl
                    level_index = i

        # Успех: разворот от глубокого уровня к предыдущему
        if current_deep_level is not None:
            if level_index == 0:
                upper_level = 0
            else:
                upper_level = levels[level_index-1]

            upper_price = level_prices[upper_level]
            if current_low <= upper_price <= current_high:
                price_start = level_prices[current_deep_level]
                price_end = level_prices[upper_level]
                price_change_abs = price_end - price_start
                price_change_pct = (price_change_abs / price_start) * 100

                profit_usd = 0.0
                commission_usd = 0.0

                # Прибыль от самого глубокого уровня
                size_deep = position_sizes[current_deep_level]
                profit_deep, comm_deep = calculate_current_level_profit_usd(
                    size_deep, price_end, price_start, commission_rate
                )
                profit_usd += profit_deep
                commission_usd += comm_deep

                # Убытки по всем более ранним уровням (закрываются на upper_level)
                for j in range(level_index):
                    lvl_j = levels[j]
                    size_j = position_sizes[lvl_j]
                    price_close_j = level_prices[upper_level]
                    price_open_j = level_prices[lvl_j]
                    loss_j, comm_j = calculate_current_level_profit_usd(
                        size_j, price_close_j, price_open_j, commission_rate
                    )
                    profit_usd += loss_j
                    commission_usd += comm_j

                return {
                    'status': 'success',
                    'level': current_deep_level,
                    'fill_start_level': current_deep_level,
                    'fill_end_level': upper_level,
                    'timestamp': idx,
                    'price_start': price_start,
                    'price_end': price_end,
                    'price_change_abs': price_change_abs,
                    'price_change_pct': price_change_pct,
                    'level_return_pct': price_change_pct,
                    'profit_usd': profit_usd,
                    'profit_pct': (profit_usd / capital_per_trade) * 100,
                    'commission_usd': commission_usd,
                    'details': f'Price reached {current_deep_level}, returned to {upper_level}'
                }

        # Стоп-лосс
        if use_stop_loss and current_low <= level_prices[sl_level] <= current_high:
            last_level = current_deep_level if current_deep_level is not None else 0
            price_start = level_prices[last_level]
            price_end = level_prices[sl_level]
            price_change_abs = price_end - price_start
            price_change_pct = (price_change_abs / price_start) * 100

            loss_usd = 0.0
            commission_usd = 0.0
            for lvl in levels:
                loss = calculate_stop_loss_level_loss_usd(level_prices, sl_level, lvl, commission_rate)
                comm = calculate_stop_loss_level_commission_usd(level_prices, sl_level, lvl, commission_rate)
                loss_usd += loss
                commission_usd += comm

            return {
                'status': 'failure',
                'level': sl_level,
                'fill_start_level': last_level,
                'fill_end_level': sl_level,
                'timestamp': idx,
                'price_start': price_start,
                'price_end': price_end,
                'price_change_abs': price_change_abs,
                'price_change_pct': price_change_pct,
                'level_return_pct': price_change_pct,
                'profit_usd': loss_usd,
                'profit_pct': (loss_usd / capital_per_trade) * 100,
                'commission_usd': commission_usd,
                'details': f'Stop Loss at {sl_level}'
            }

    return {'status':'in_progress','level':None,'timestamp':None,
            'fill_start_level':None,'fill_end_level':None,
            'profit_usd':0,'profit_pct':0,'level_return_pct':0,
            'commission_usd':0,'details':'Analysis not completed'}

def collect_statistics(df, df_1m, target_indices, trading_timeframe):
    """
    Собирает статистику по всем целевым свечам, включая комиссии.
    Учитывает только завершённые сделки (success и failure) для расчёта средней комиссии.
    """
    stats = {
        'total': len(target_indices),
        'success': 0,
        'failure': 0,
        'in_progress': 0,
        'by_level': {},
        'by_level_avg_pct': {},
        'by_level_avg_abs': {},
        'by_level_avg_profit_usd': {},
        'by_level_avg_profit_pct': {},
        'stop_loss': 0,
        'total_avg_pct': 0,
        'total_avg_abs': 0,
        'total_avg_profit_usd': 0,
        'total_avg_profit_pct': 0,
        'total_avg_loss_usd': 0,
        'total_avg_loss_pct': 0,
        'total_profit_sum': 0,
        'total_loss_sum': 0,
        'net_profit': 0,
        'profit_factor': None,
        'total_commission_usd': 0,
        'avg_commission_per_trade_usd': 0
    }

    # Временные хранилища
    level_pct_sum = {}
    level_abs_sum = {}
    level_count = {}
    level_profit_usd_sum = {}
    level_profit_pct_sum = {}

    total_pct_sum = 0
    total_abs_sum = 0
    total_profit_usd_sum = 0
    total_success_count = 0
    total_loss_usd_sum = 0
    total_loss_count = 0
    total_commission = 0.0
    total_completed_trades = 0

    target_indices_len = len(target_indices)
    i = 0

    for idx in target_indices:
        i += 1
        print(f"\r{i} of {target_indices_len} target candles. Running time: {format_duration(time.perf_counter() - start_time)}", end='', flush=True)
        outcome = analyze_target_candle_outcome(df, df_1m, idx, trading_timeframe)

        if outcome['status'] == 'success':
            stats['success'] += 1
            level = outcome['level']
            pct_change = outcome['price_change_pct']
            abs_change = outcome['price_change_abs']
            profit_usd = outcome.get('profit_usd', 0)
            profit_pct = outcome.get('profit_pct', 0)
            commission = outcome.get('commission_usd', 0)

            total_commission += commission
            total_completed_trades += 1

            if level not in stats['by_level']:
                stats['by_level'][level] = 0
                level_pct_sum[level] = 0
                level_abs_sum[level] = 0
                level_profit_usd_sum[level] = 0
                level_profit_pct_sum[level] = 0
                level_count[level] = 0
            stats['by_level'][level] += 1
            level_pct_sum[level] += pct_change
            level_abs_sum[level] += abs_change
            level_profit_usd_sum[level] += profit_usd
            level_profit_pct_sum[level] += profit_pct
            level_count[level] += 1

            total_pct_sum += pct_change
            total_abs_sum += abs_change
            total_profit_usd_sum += profit_usd
            total_success_count += 1

        elif outcome['status'] == 'failure':
            stats['failure'] += 1
            stats['stop_loss'] += 1
            loss_usd = outcome.get('profit_usd', 0)
            commission = outcome.get('commission_usd', 0)

            total_commission += commission
            total_completed_trades += 1

            total_loss_usd_sum += loss_usd
            total_loss_count += 1

        elif outcome['status'] == 'in_progress':
            stats['in_progress'] += 1

    # Средние по уровням (только для успешных)
    for level in level_count:
        stats['by_level_avg_pct'][level] = level_pct_sum[level] / level_count[level]
        stats['by_level_avg_abs'][level] = level_abs_sum[level] / level_count[level]
        stats['by_level_avg_profit_usd'][level] = level_profit_usd_sum[level] / level_count[level]
        stats['by_level_avg_profit_pct'][level] = level_profit_pct_sum[level] / level_count[level]

    if total_success_count > 0:
        stats['total_avg_pct'] = total_pct_sum / total_success_count
        stats['total_avg_abs'] = total_abs_sum / total_success_count
        stats['total_avg_profit_usd'] = total_profit_usd_sum / total_success_count
        stats['total_avg_profit_pct'] = (total_profit_usd_sum / (total_success_count * capital_per_trade)) * 100

    if total_loss_count > 0:
        stats['total_avg_loss_usd'] = total_loss_usd_sum / total_loss_count
        stats['total_avg_loss_pct'] = (total_loss_usd_sum / (total_loss_count * capital_per_trade)) * 100

    stats['total_profit_sum'] = total_profit_usd_sum
    stats['total_loss_sum'] = total_loss_usd_sum
    stats['net_profit'] = total_profit_usd_sum + total_loss_usd_sum

    if total_loss_usd_sum != 0:
        stats['profit_factor'] = abs(total_profit_usd_sum / total_loss_usd_sum)

    stats['total_commission_usd'] = total_commission
    if total_completed_trades > 0:
        stats['avg_commission_per_trade_usd'] = total_commission / total_completed_trades

    stats['duration'] = time.perf_counter() - start_time

    return stats

def print_statistics(stats):
    print("\n" + "="*60)
    print("📊 TARGET CANDLE STATISTICS")
    print("="*60)
    print(f"\n📌 GENERAL STATISTICS:")
    print(f"   Symbol: {symbol}")
    print(f"   Timeframe: {discretization}")
    print(f"   Start date: {display_start_date_str}")
    print(f"   End date: {load_end_date.strftime('%Y-%m-%d %H:%M')}")
    print(f"   Duration: {format_duration(stats['duration'])}")
    print(f"   Total target candles: {stats['total']}")
    print(f"   Measure percentile: {measure_percentile}")
    print(f"   Use candle size instead of shadow: {use_candle_size_instead_of_shadow}")
    print(f"   Use MRC: {use_mrc}")
    if use_mrc:
        print(f"   Use MRC R2: {use_mrc_r2}")
        print(f"   Use MRC S2: {use_mrc_s2}")
    print(f"   Use SL: {use_stop_loss}")
    if use_stop_loss:
        print(f"   SL level: {sl_level}")
    print(f"\n📊 POSITION SIZES:")
    for level, size in position_sizes.items():
        print(f"   Level {level:.3f}: ${size:.2f} ({size/capital_per_trade*100:.1f}%)")
    print(f"\n💰 CAPITAL & FEES:")
    print(f"   Capital per trade: ${capital_per_trade}")
    print(f"   Commission rate: {commission_rate*100:.3f}% per transaction")
    print(f"\n📈 RESULTS:")
    completed = stats['success'] + stats['failure']
    print(f"   ✅ SUCCESSFUL: {stats['success']} ({stats['success']/stats['total']*100:.1f}% of total, {stats['success']/completed*100:.1f}% of completed)" if completed else "   ✅ SUCCESSFUL: 0")
    print(f"   ❌ UNSUCCESSFUL: {stats['failure']} ({stats['failure']/stats['total']*100:.1f}% of total, {stats['failure']/completed*100:.1f}% of completed)" if completed else "   ❌ UNSUCCESSFUL: 0")
    print(f"   🔄 IN PROGRESS: {stats['in_progress']} ({stats['in_progress']/stats['total']*100:.1f}%)")
    # остальные строки без изменений...
    print(f"\n🎯 SUCCESS BY LEVEL:")
    for level in reversed(sorted(stats['by_level'].keys())):
        count = stats['by_level'][level]
        avg_usd = stats['by_level_avg_profit_usd'][level]
        avg_pct = stats['by_level_avg_profit_pct'][level]
        print(f"   Level {level:.3f}: {count} ({count/stats['success']*100:.1f}% of successful) | 📊 Avg profit: ${avg_usd:.2f} ({avg_pct:.2f}%)")
    print(f"\n📊 AVERAGE SUCCESS METRICS:")
    print(f"   📈 Avg profit per success: ${stats['total_avg_profit_usd']:.2f} ({stats['total_avg_profit_pct']:.2f}%)")
    print(f"\n🛑 FAILURES:")
    print(f"   Stop Loss: {stats['stop_loss']}")
    if stats['total_avg_loss_usd'] != 0:
        print(f"   📉 Avg loss per failure: ${stats['total_avg_loss_usd']:.2f} ({stats['total_avg_loss_pct']:.2f}%)")
    print(f"\n💰 FINANCIAL SUMMARY:")
    print(f"   Total profit: ${stats['total_profit_sum']:.2f}")
    print(f"   Total loss: ${stats['total_loss_sum']:.2f}")
    print(f"   Net profit: ${stats['net_profit']:.2f}")
    if stats['profit_factor'] is not None:
        print(f"   Profit Factor: {stats['profit_factor']:.2f}")
    else:
        print(f"   Profit Factor: N/A (no losses)")
    print(f"\n💸 COMMISSIONS:")
    print(f"   Total commissions paid: ${stats['total_commission_usd']:.2f}")
    if stats['total'] > 0:
        print(f"   Avg commission per trade: ${stats['avg_commission_per_trade_usd']:.2f}")
    print("\n" + "="*60)

def analyze_and_print_statistics(df, df_1m, trading_timeframe):
    target_candle_indices = df[df['is_target_candle']].index.tolist()
    if not target_candle_indices:
        print("There are no target candles for analysis")
        return
    stats = collect_statistics(df, df_1m, target_candle_indices, trading_timeframe)
    print_statistics(stats)
    return stats

In [ ]:
import ta
import numpy as np
import pandas as pd
from SRC.CORE.utils import featurize_lambda
from SRC.CORE._CONSTANTS import _UTC_TIMESTAMP
#Process iteration over timeseries dataframe with sliding window approach
window_size = 30

def mrc_supersmoother(src, length):
    """
    Supersmoother filter by John Ehlers
    """
    a1 = np.exp(-1.414 * np.pi / length)
    b1 = 2 * a1 * np.cos(1.414 * np.pi / length)
    c3 = -a1 * a1
    c2 = b1
    c1 = 1 - c2 - c3

    ss = np.zeros_like(src)
    ss[:2] = src[:2]

    for i in range(2, len(src)):
        ss[i] = c1 * src[i] + c2 * ss[i-1] + c3 * ss[i-2]

    return ss

def mrc_sak_filter(filter_type, src, length):
    """
    Ehlers Swiss Army Knife filters
    """
    pi = np.pi
    cycle = 2 * pi / length

    c0, c1 = 1.0, 0.0
    b0, b1, b2 = 1.0, 0.0, 0.0
    a1, a2 = 0.0, 0.0
    alpha, beta, gamma = 0.0, 0.0, 0.0

    if filter_type == "Ehlers EMA":
        alpha = (np.cos(cycle) + np.sin(cycle) - 1) / np.cos(cycle)
        b0 = alpha
        a1 = 1 - alpha

    elif filter_type == "Gaussian":
        beta = 2.415 * (1 - np.cos(cycle))
        alpha = -beta + np.sqrt(beta * beta + 2 * beta)
        c0 = alpha * alpha
        a1 = 2 * (1 - alpha)
        a2 = -(1 - alpha) * (1 - alpha)

    elif filter_type == "Butterworth":
        beta = 2.415 * (1 - np.cos(cycle))
        alpha = -beta + np.sqrt(beta * beta + 2 * beta)
        c0 = alpha * alpha / 4
        b1, b2 = 2, 1
        a1 = 2 * (1 - alpha)
        a2 = -(1 - alpha) * (1 - alpha)

    elif filter_type == "BandStop":
        beta = np.cos(cycle)
        gamma = 1 / np.cos(cycle * 2 * 0.1)  # delta = 0.1
        alpha = gamma - np.sqrt(gamma * gamma - 1)
        c0 = (1 + alpha) / 2
        b1 = -2 * beta
        b2 = 1
        a1 = beta * (1 + alpha)
        a2 = -alpha

    elif filter_type == "SMA":
        c1 = 1 / length
        b0 = 1 / length
        a1 = 1

    elif filter_type == "EMA":
        alpha = 2 / (length + 1)
        b0 = alpha
        a1 = 1 - alpha

    elif filter_type == "RMA":
        alpha = 1 / length
        b0 = alpha
        a1 = 1 - alpha

    output = np.zeros_like(src)
    output[:3] = src[:3]

    for i in range(3, len(src)):
        output[i] = (c0 * (b0 * src[i] +
                          b1 * (src[i-1] if i-1 >= 0 else 0) +
                          b2 * (src[i-2] if i-2 >= 0 else 0)) +
                    a1 * output[i-1] +
                    a2 * output[i-2] -
                    c1 * (src[i-length] if i-length >= 0 else 0))

    return output

def mrc_calculate(source_type='hlc3', filter_type='SuperSmoother', innermult=1.0, outermult=2.415):
    """
    Calculate Mean Reversion Channel

    Использует глобальные переменные:
    - mrc_df: DataFrame с данными (включая буфер)
    - df: DataFrame для отображения (без буфера)
    """
    global mrc_df, df

    # Calculate source price на mrc_df (с буфером)
    if source_type == 'hlc3':
        source = (mrc_df['high'] + mrc_df['low'] + mrc_df['close']) / 3
    elif source_type == 'ohlc4':
        source = (mrc_df['open'] + mrc_df['high'] + mrc_df['low'] + mrc_df['close']) / 4
    else:  # 'close'
        source = mrc_df['close']

    source = source.values

    # Calculate True Range на mrc_df
    tr = np.maximum(
        mrc_df['high'] - mrc_df['low'],
        np.maximum(
            abs(mrc_df['high'] - mrc_df['close'].shift(1)),
            abs(mrc_df['low'] - mrc_df['close'].shift(1))
        )
    ).fillna(0).values

    length = 200

    # Apply filters
    if filter_type == 'SuperSmoother':
        meanline = mrc_supersmoother(source, length)
        meanrange = mrc_supersmoother(tr, length)
    else:
        meanline = mrc_sak_filter(filter_type, source, length)
        meanrange = mrc_sak_filter(filter_type, tr, length)

    pi = np.pi
    mult = pi * innermult
    mult2 = pi * outermult

    # Calculate channels
    upband1 = meanline + (meanrange * mult)
    loband1 = meanline - (meanrange * mult)
    upband2 = meanline + (meanrange * mult2)
    loband2 = meanline - (meanrange * mult2)

    # Сохраняем результаты в mrc_df
    mrc_df['meanline'] = meanline
    mrc_df['meanrange'] = meanrange
    mrc_df['upband1'] = upband1
    mrc_df['loband1'] = loband1
    mrc_df['upband2'] = upband2
    mrc_df['loband2'] = loband2

    # Переносим результаты из mrc_df в df (видимый диапазон)
    df['meanline'] = mrc_df.loc[df.index, 'meanline']
    df['meanrange'] = mrc_df.loc[df.index, 'meanrange']
    df['upband1'] = mrc_df.loc[df.index, 'upband1']
    df['loband1'] = mrc_df.loc[df.index, 'loband1']
    df['upband2'] = mrc_df.loc[df.index, 'upband2']
    df['loband2'] = mrc_df.loc[df.index, 'loband2']

def stochastic_tradingview(high, low, close, periodK=14, smoothK=3, periodD=3):
    lowest_low = low.rolling(window=periodK).min()
    highest_high = high.rolling(window=periodK).max()
    raw_k = 100 * (close - lowest_low) / (highest_high - lowest_low)
    stoch_k = raw_k.rolling(window=smoothK).mean()
    stoch_d = stoch_k.rolling(window=periodD).mean()

    return stoch_k, stoch_d

def find_target_candles(df_display, df_buffer, window=10):
    """
    Поиск целевых свечей по критериям:
    1. Касание линии R2 или S2 тенью/телом
    2. Объем выше перцентиля предыдущих свечей
    3. Размер свечи/тень выше перцентиля предыдущих свечей
    """
    if df_display.index[0] in df_buffer.index:
        df_full = df_buffer.copy()
    else:
        df_full = pd.concat([df_buffer, df_display]).drop_duplicates().sort_index()

    # Скользящие перцентили для объёма (по предыдущим свечам)
    df_full['volume_percentile'] = df_full['volume'].shift(1).rolling(window=window, min_periods=1).quantile(measure_percentile)

    # В зависимости от флага рассчитываем меру и её перцентиль
    if use_candle_size_instead_of_shadow:
        df_full['measure'] = df_full['high'] - df_full['low']
    else:
        df_full['upper_shadow'] = df_full['high'] - df_full[['open', 'close']].max(axis=1)
        df_full['lower_shadow'] = df_full[['open', 'close']].min(axis=1) - df_full['low']
        df_full['measure'] = df_full[['upper_shadow', 'lower_shadow']].max(axis=1)

    df_full['measure_percentile'] = df_full['measure'].shift(1).rolling(window=window, min_periods=1).quantile(measure_percentile)

    # Поиск целевых свечей
    target_conditions = []
    for idx in df_display.index:
        i = df_full.index.get_loc(idx)
        if isinstance(i, (slice, np.ndarray)):
            i = i.start if isinstance(i, slice) else i[0]

        if i < window:
            target_conditions.append(False)
            continue

        high = df_full['high'].iloc[i]
        low = df_full['low'].iloc[i]
        r2 = df_full['upband2'].iloc[i]
        s2 = df_full['loband2'].iloc[i]

        touches_r2 = (high >= r2 and low <= r2) or (low > r2)
        touches_s2 = (high >= s2 and low <= s2) or (high < s2)
        touches_level = not use_mrc or (use_mrc_r2 and touches_r2) or (use_mrc_s2 and touches_s2)

        volume_above = df_full['volume'].iloc[i] > df_full['volume_percentile'].iloc[i]
        measure_above = df_full['measure'].iloc[i] > df_full['measure_percentile'].iloc[i]

        target_conditions.append(touches_level and volume_above and measure_above)

    df_display['is_target_candle'] = target_conditions

    return df_display

def get_display_timeframe(trading_timeframe):
    """
    Возвращает таймфрейм для отображения графика в зависимости от торгового таймфрейма
    """
    mapping = {
        '1D': '1H',
        '8H': '30M',
        '4H': '15M',
        '1H': '5M',
        '30M': '1M',
        '15M': '1M',
        '5M': '1M'
    }
    return mapping.get(trading_timeframe, '1M')

def resample_to_timeframe(df_1m, timeframe):
    """
    Преобразует 1-минутные данные в указанный таймфрейм
    """
    interval_minutes = {
        '1M': 1,
        '5M': 5,
        '15M': 15,
        '30M': 30,
        '1H': 60,
        '4H': 240,
        '8H': 480,
        '1D': 1440
    }.get(timeframe, 60)

    if timeframe == '1M':
        return df_1m

    # Убираем часовой пояс
    if df_1m.index.tz is not None:
        df_no_tz = df_1m.tz_localize(None)
        original_tz = df_1m.index.tz
    else:
        df_no_tz = df_1m
        original_tz = None

    # Используем стандартный resample вместо groupby
    rule = f'{interval_minutes}T'
    df_resampled = df_no_tz.resample(rule, closed='left', label='left').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum'
    }).dropna()

    # Возвращаем часовой пояс
    if original_tz is not None:
        df_resampled.index = df_resampled.index.tz_localize(original_tz)

    df_resampled[_UTC_TIMESTAMP] = df_resampled.index
    df_resampled = featurize_lambda(df_resampled, _UTC_TIMESTAMP, _KIEV_TIMESTAMP, lambda utc_ts: as_kiev_tz(utc_ts))

    return df_resampled

# Убеждаемся, что индексы уникальны
mrc_df = mrc_df[~mrc_df.index.duplicated(keep='first')]
df = df[~df.index.duplicated(keep='first')]

# Функция для подготовки данных с буфером
def prepare_buffer_data(df_main, df_display, buffer_size):
    combined = pd.concat([df_main.iloc[-(buffer_size + len(df_display)):], df_display])
    combined = combined[~combined.index.duplicated(keep='last')].sort_index()
    return combined

# 1. RSI
rsi_calc_df = prepare_buffer_data(mrc_df, df, rsi_buffer)
df['rsi'] = ta.momentum.RSIIndicator(close=rsi_calc_df['close'], window=14).rsi().loc[df.index]

# 2. Stochastic
stoch_calc_df = prepare_buffer_data(mrc_df, df, stoch_buffer)
stoch_k_full, stoch_d_full = stochastic_tradingview(
    high=stoch_calc_df['high'],
    low=stoch_calc_df['low'],
    close=stoch_calc_df['close']
)
df['stoch_k'] = stoch_k_full.loc[df.index]
df['stoch_d'] = stoch_d_full.loc[df.index]

# 3. MACD
macd_calc_df = prepare_buffer_data(mrc_df, df, macd_buffer)
macd = ta.trend.MACD(
    close=macd_calc_df['close'],
    window_slow=26,
    window_fast=12,
    window_sign=9
)
df['macd_line'] = macd.macd().loc[df.index]
df['macd_signal'] = macd.macd_signal().loc[df.index]
df['macd_histogram'] = macd.macd_diff().loc[df.index]

# 4. ATR
atr_calc_df = prepare_buffer_data(mrc_df, df, atr_buffer)
atr_full = ta.volatility.AverageTrueRange(
    high=atr_calc_df['high'],
    low=atr_calc_df['low'],
    close=atr_calc_df['close'],
    window=14
).average_true_range()
df['atr'] = atr_full.loc[df.index]

In [ ]:
%load_ext autoreload
%autoreload

from LIBRARIES.time_utils import as_kiev_tz
import plotly.graph_objects as go
from plotly.subplots import make_subplots

is_log_scale_by_default=True
candlestick_row = 1
volume_row = 2
rsi_row = 3
stoch_row = 4
macd_row = 5
atr_row = 6

def add_bars(col, name, color, row):
    fig_main.add_trace(
        go.Bar(
            x=df[_KIEV_TIMESTAMP],
            y=df[col],
            name=name,
            marker=dict(color=color),
            width=(df.index[1] - df.index[0]).total_seconds() * 1000,
        ),
        row=row, col=1
)

def add_scatter(col, name, color, row, fill=None, fillcolor=None, width=2, dash=None):
    fig_main.add_trace(
        go.Scatter(
            x=df[_KIEV_TIMESTAMP],
            y=df[col],
            name=name,
            line=dict(color=color, width=width, dash=dash),
            mode='lines',
            fill=fill,
            fillcolor=fillcolor
        ),
        row=row, col=1
    )

def add_target_candle_scatter(position, position_multiplier, name, color, symbol_direction):
    signals = df[df['is_target_candle']]

    if len(signals) > 0:
        fig_main.add_trace(
            go.Scatter(
                x=signals[_KIEV_TIMESTAMP],
                y=signals[position] * position_multiplier,
                name=name + " TC",
                mode='markers',
                marker=dict(color=color, size=10, symbol='triangle-' + symbol_direction)
            ),
            row=candlestick_row, col=1
        )

def add_over_zone(y0, y1, fillcolor, row):
    fig_main.add_hrect(
        y0=y0, y1=y1,
        line_width=0,
        fillcolor=fillcolor,
        opacity=0.2,
        row=row, col=1
    )

def add_central_line(row, y=50, line_dash="dot"):
    fig_main.add_hline(
        y=y,
        line_dash=line_dash,
        line_color="white",
        line_width=1,
        opacity=0.3,
        row=row, col=1
    )

def add_over_zones_and_a_central_line(row):
    add_over_zone(80, 100, "red", row)
    add_over_zone(0, 20, "green", row)
    add_central_line(row)

def add_stoch(speed, color):
    add_scatter('stoch_' + speed, "Stoch %" + speed.capitalize(), color, stoch_row)

def add_fibonacci_levels(fig, target_data, target_idx, df_subset, row=1, col=1, fill_start_level=None, fill_end_level=None, fill_status=None):
    import math
    high = target_data['high'].iloc[0]
    low = target_data['low'].iloc[0]

    log_start = math.log(low)
    log_end = math.log(high)
    log_range = log_end - log_start

    # Базовые уровни: 0, 1 и все торговые уровни + стоп-лосс
    all_display_levels = [0, 1] + levels + [sl_level]
    # Цвета (можно настроить)
    colors_map = {0: 'yellow', 1: 'yellow'}
    color_palette = ['violet', 'lavender', 'orange', 'pink', 'aqua']
    for i, lvl in enumerate(levels):
        colors_map[lvl] = color_palette[i % len(color_palette)]
    colors_map[sl_level] = 'aqua'

    prices = {lvl: math.exp(log_start + log_range * lvl) for lvl in all_display_levels}

    if len(df_subset.index) >= 2:
        time_offset = df_subset.index[0] - (df_subset.index[1] - df_subset.index[0])
    else:
        time_offset = df_subset.index[0] - pd.Timedelta(hours=1)

    end_date = df_subset.index[-1]

    # Заливка (если есть)
    if fill_start_level is not None and fill_end_level is not None:
        if fill_start_level in prices and fill_end_level in prices:
            y_lower = min(prices[fill_start_level], prices[fill_end_level])
            y_upper = max(prices[fill_start_level], prices[fill_end_level])
            fill_color = 'red' if fill_status == 'failure' else colors_map.get(fill_start_level, 'gray')
            fig.add_trace(
                go.Scatter(
                    x=[as_kiev_tz(target_idx), as_kiev_tz(end_date), as_kiev_tz(end_date), as_kiev_tz(target_idx)],
                    y=[y_lower, y_lower, y_upper, y_upper],
                    fill='toself',
                    mode='lines',
                    name=f"Fill {fill_start_level}→{fill_end_level}",
                    line=dict(width=0),
                    fillcolor=fill_color,
                    opacity=0.3,
                    showlegend=False
                ),
                row=row, col=col
            )

    # Линии и текст
    for lvl, p in prices.items():
        fig.add_trace(
            go.Scatter(
                x=[as_kiev_tz(target_idx), as_kiev_tz(end_date)],
                y=[p, p],
                mode='lines',
                name=f"Fib {lvl:.3f}: {p:.2f}",
                line=dict(color=colors_map[lvl], width=1),
                showlegend=True
            ),
            row=row, col=col
        )
        fig.add_trace(
            go.Scatter(
                x=[as_kiev_tz(time_offset)],
                y=[p],
                mode='text',
                text=[f"{lvl:.3f} ({p:.2f})"],
                textposition="middle left",
                textfont=dict(size=12, color=colors_map[lvl], family="Arial"),
                showlegend=False,
                hoverinfo='none'
            ),
            row=row, col=col
        )

    return fig

def plot_target_candle_with_fib_multiframe(df_trading, df_1m, target_idx, future_bars, trading_timeframe):
    """
    Строит график для целевой свечи
    """
    import math

    display_timeframe = get_display_timeframe(trading_timeframe)
    df_display = resample_to_timeframe(df_1m, display_timeframe)

    # Определяем начало отображения в зависимости от торгового ТФ
    if trading_timeframe == '1D':
        next_day = target_idx + pd.Timedelta(days=1)
        mask = df_display.index >= next_day
        if mask.any():
            target_idx_display = df_display.index[mask][0]
        else:
            target_idx_display = target_idx
    else:
        interval_minutes = {
            '1M': 1, '5M': 5, '15M': 15, '30M': 30, '1H': 60, '4H': 240, '8H': 480
        }.get(trading_timeframe, 1)
        candle_close = target_idx + pd.Timedelta(minutes=interval_minutes)
        mask = df_display.index >= candle_close
        if mask.any():
            target_idx_display = df_display.index[mask][0]
        else:
            target_idx_display = target_idx

    if target_idx_display not in df_display.index:
        target_pos = df_display.index.get_indexer([target_idx_display], method='nearest')[0]
        target_idx_display = df_display.index[target_pos]
        print(f"Adjusted target_idx_display: {target_idx_display}")

    target_data = df_trading.loc[[target_idx]]
    outcome = analyze_target_candle_outcome(df_trading, df_1m, target_idx, trading_timeframe)

    # Определяем диапазон для отображения
    if outcome['status'] in ['success', 'failure'] and outcome['timestamp'] is not None:
        result_ts = outcome['timestamp']

        if df_display.index.tz is not None and result_ts.tz is None:
            result_ts = result_ts.tz_localize(df_display.index.tz)
        elif df_display.index.tz is None and result_ts.tz is not None:
            result_ts = result_ts.tz_localize(None)

        df_after_target = df_display.loc[target_idx_display:]
        mask = df_after_target.index <= result_ts
        if mask.any():
            end_dt = df_after_target.index[mask][-1]
        else:
            end_dt = target_idx_display
    else:
        target_pos = df_display.index.get_loc(target_idx_display) if target_idx_display in df_display.index else 0
        end_pos = min(len(df_display) - 1, target_pos + future_bars)
        end_dt = df_display.index[end_pos]

    if end_dt <= target_idx_display:
        next_pos = df_display.index.get_loc(target_idx_display) + 1
        if next_pos < len(df_display):
            end_dt = df_display.index[next_pos]
        else:
            end_dt = target_idx_display

    df_subset = df_display.loc[target_idx_display:end_dt].copy()

    if len(df_subset) < 2:
        current_pos = df_display.index.get_loc(end_dt)
        next_pos = current_pos + 1
        if next_pos < len(df_display):
            end_dt = df_display.index[next_pos]
            df_subset = df_display.loc[target_idx_display:end_dt].copy()
            print(f"Extended df_subset to {len(df_subset)} candles (was 1)")

    fig_target_candle = make_subplots(rows=1, cols=1)

    fig_target_candle.add_trace(
        go.Candlestick(
            x=df_subset[_KIEV_TIMESTAMP],
            open=df_subset["open"],
            high=df_subset["high"],
            low=df_subset["low"],
            close=df_subset["close"],
            name=f"OHLC ({display_timeframe})",
            showlegend=True
        ),
        row=1, col=1
    )

    # Информация о комиссии
    commission_usd = outcome.get('commission_usd', 0)
    commission_info = f" | 💸 Commission: ${commission_usd:.2f} (Parameter: {commission_rate*100:.3f}%)"

    date_str = as_kiev_tz(target_idx).strftime('%Y-%m-%d %H:%M') if hasattr(as_kiev_tz(target_idx), 'strftime') else str(as_kiev_tz(target_idx))

    if outcome['status'] == 'success':
        result_str = f"✅ SUCCESS (level: {outcome['level']:.3f})"
        result_time = as_kiev_tz(outcome['timestamp']).strftime('%Y-%m-%d %H:%M') if hasattr(as_kiev_tz(outcome['timestamp']), 'strftime') else str(as_kiev_tz(outcome['timestamp']))
        level_return = outcome.get('level_return_pct', 0)
        profit_info = f" | 💰 Profit: ${outcome['profit_usd']:.2f} ({outcome['profit_pct']:.2f}%)"
        title = f"Trading TF: {trading_timeframe} | Display TF: {display_timeframe} | 🎯 TC: {date_str} | {result_str} | 📈 Level return: {level_return:.2f}% | 📍 Result: {result_time}{profit_info}{commission_info}"

    elif outcome['status'] == 'failure':
        result_str = f"❌ FAILURE (SL: {outcome['level']:.3f})"
        result_time = as_kiev_tz(outcome['timestamp']).strftime('%Y-%m-%d %H:%M') if hasattr(as_kiev_tz(outcome['timestamp']), 'strftime') else str(as_kiev_tz(outcome['timestamp']))
        level_return = outcome.get('level_return_pct', 0)
        loss_info = f" | 💰 Loss: ${outcome['profit_usd']:.2f} ({outcome['profit_pct']:.2f}%)"
        title = f"Trading TF: {trading_timeframe} | Display TF: {display_timeframe} | 🎯 TC: {date_str} | {result_str} | 📈 Level return: {level_return:.2f}% | 📍 Result: {result_time}{loss_info}{commission_info}"

    elif outcome['status'] == 'in_progress':
        title = f"Trading TF: {trading_timeframe} | Display TF: {display_timeframe} | 🎯 TC: {date_str} | 🔄 IN PROGRESS"
    else:
        title = f"Trading TF: {trading_timeframe} | Display TF: {display_timeframe} | 🎯 TC: {date_str} | ⚪ UNCERTAIN"

    fig_target_candle = add_fibonacci_levels(
        fig_target_candle, target_data, target_idx_display, df_subset, row=1, col=1,
        fill_start_level=outcome.get('fill_start_level'),
        fill_end_level=outcome.get('fill_end_level'),
        fill_status=outcome['status'],
    )

    if sl_level is not None:
        high = target_data['high'].iloc[0]
        low = target_data['low'].iloc[0]
        log_start = math.log(low)
        log_end = math.log(high)

        log_range = log_end - log_start
        sl_price = math.exp(log_start + log_range * sl_level)

        fig_target_candle.add_trace(
            go.Scatter(
                x=[as_kiev_tz(target_idx_display), as_kiev_tz(df_subset.index[-1])],
                y=[sl_price, sl_price],
                mode='lines',
                name=f"🛑 Stop Loss ({sl_level})",
                line=dict(color='crimson', width=2, dash='dash'),
                showlegend=True
            ),
            row=1, col=1
        )

        fig_target_candle.add_trace(
            go.Scatter(
                x=[as_kiev_tz(target_idx_display)],
                y=[sl_price],
                mode='text',
                text=[f"🛑"],
                textposition="middle right",
                textfont=dict(size=11, color='red', family="Arial"),
                showlegend=False,
                hoverinfo='none'
            ),
            row=1, col=1
        )

    if outcome['status'] in ['success', 'failure'] and outcome['timestamp'] is not None:
        result_dt = outcome['timestamp']

        if df_subset.index.tz is not None and result_dt.tz is None:
            result_dt = result_dt.tz_localize(df_subset.index.tz)
        elif df_subset.index.tz is None and result_dt.tz is not None:
            result_dt = result_dt.tz_localize(None)

        indices = df_subset.index.tolist()
        target_candle_idx = None
        for idx in indices:
            if idx <= result_dt:
                target_candle_idx = idx
            else:
                break

        if target_candle_idx is not None:
            fig_target_candle.add_vline(
                x=as_kiev_tz(target_candle_idx),
                line_dash="dash",
                line_color='lime' if outcome['status'] == 'success' else 'red',
                line_width=0.4,
                opacity=1,
                row=1, col=1
            )

    fig_target_candle.update_layout(
        title=title,
        xaxis_rangeslider_visible=False,
        yaxis_type="log",
        yaxis_title="Price (log scale)",
        template="plotly_dark",
        height=1700,
        showlegend=True,
        hovermode='x unified'
    )

    return fig_target_candle

def plot_all_target_candles_multiframe_and_save_htmls_to_disc(df_trading, df_1m, trading_timeframe, future_bars=1000, output_dir="target_candles", clean_dir=True):
    """
    Сохраняет графики для всех целевых свечей с учетом мультитаймфрейма
    """
    import shutil
    import os

    target_candle_indices = df_trading[df_trading['is_target_candle']].index.tolist()

    if not target_candle_indices:
        print("No target candles to display")
        return

    if clean_dir and os.path.exists(output_dir):
        print(f"Cleaning folder '{output_dir}'...")
        shutil.rmtree(output_dir)
        print(f"Folder '{output_dir}' deleted")

    os.makedirs(output_dir, exist_ok=True)
    print(f"Folder '{output_dir}' created")
    print(f"Trading timeframe: {trading_timeframe}")
    print(f"Display timeframe: {get_display_timeframe(trading_timeframe)}")
    print(f"Total target candles: {len(target_candle_indices)}")

    for i, target_idx in enumerate(target_candle_indices):
        date_str = as_kiev_tz(target_idx).strftime('%Y-%m-%d_%H-%M') if hasattr(as_kiev_tz(target_idx), 'strftime') else str(as_kiev_tz(target_idx))
        filename = f"{output_dir}/target_{i+1:03d}_{trading_timeframe}_{date_str}.html"

        fig = plot_target_candle_with_fib_multiframe(
            df_trading, df_1m, target_idx, future_bars, trading_timeframe
        )
        fig.write_html(filename)

    print(f"\n✅ All {len(target_candle_indices)} charts saved to folder '{output_dir}'")
    print(f"   Open the files in your browser to view")

fig_main = make_subplots(
    rows=6, cols=1,
    shared_xaxes=True,
    row_heights=[3, 0.7, 1, 1, 1.5, 1],
    vertical_spacing=0.03
)

fig_main.add_trace(
    go.Candlestick(
        x=df[_KIEV_TIMESTAMP],
        open=df["open"],
        high=df["high"],
        low=df["low"],
        close=df["close"],
        name="OHLC"
    ),
    row=candlestick_row, col=1
)

mrc_calculate()
add_scatter('meanline', "MRC Mean", '#FFCD00', candlestick_row)
add_scatter('upband1', "MRC R1", 'green', candlestick_row, width=1, dash='dot')
add_scatter('loband1', "MRC S1", 'green', candlestick_row, width=1, dash='dot')
add_scatter('upband2', "MRC R2", 'red', candlestick_row, width=1)
add_scatter('loband2', "MRC S2", 'red', candlestick_row, width=1)

add_bars(
    "volume",
    "Volume",
    ["green" if c > o else "red" for o, c in zip(df["open"], df["close"])],
    volume_row)

add_scatter('rsi', "RSI", 'purple', rsi_row)
add_over_zones_and_a_central_line(rsi_row)

add_stoch('k', "lightblue")
add_stoch('d', "orange")
add_over_zones_and_a_central_line(stoch_row)

full_histogram = macd.macd_diff()
prev_histogram = full_histogram.shift(1).loc[df.index]
conditions = [
    (df['macd_histogram'] >= 0) & (df['macd_histogram'] >= prev_histogram),
    (df['macd_histogram'] >= 0) & (df['macd_histogram'] < prev_histogram),
    (df['macd_histogram'] < 0) & (df['macd_histogram'] <= prev_histogram),
    (df['macd_histogram'] < 0) & (df['macd_histogram'] > prev_histogram)
]
choices = ['green', 'lightgreen', 'red', 'lightcoral']
macd_colors = np.select(conditions, choices, default='rgba(128, 128, 128, 0.3)')
add_scatter("macd_line", "MACD Line", 'lightblue', macd_row)
add_scatter("macd_signal", "Signal Line", 'orange', macd_row)
add_bars("macd_histogram", "MACD Histogram", macd_colors, macd_row)
add_central_line(macd_row, line_dash="solid")

add_scatter('atr', "ATR (14)", 'orange', atr_row, fill='tozeroy', fillcolor='rgba(255, 165, 0, 0.1)')
add_central_line(atr_row, y=df['atr'].mean())

df = find_target_candles(df, mrc_df)
add_target_candle_scatter('high', 1.005, "🐵", 'white', "down")

price_log_scale_value="log"
price_linear_scale_value="linear"
price_log_scale_title="Price (log scale)"
price_linear_scale_title="Price (linear scale)"

fig_main.update_layout(
    title=f"🐵{discretization}🐵",
    xaxis_rangeslider_visible=False,
    yaxis1_type=price_log_scale_value if is_log_scale_by_default else price_linear_scale_value,
    yaxis1_title=price_log_scale_title if is_log_scale_by_default else price_linear_scale_title,
    yaxis2_title="Volume",
    yaxis3_title="RSI",
    yaxis4_title="Stoch",
    yaxis5_title="MACD",
    yaxis6_title="ATR",
    height=1200,
    bargap=0,
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            active=1 if is_log_scale_by_default else 0,
            x=0.115,
            y=1.073,
            buttons=[
                dict(
                    label="Linear",
                    method="relayout",
                    args=[{"yaxis.type": price_linear_scale_value, "yaxis.title.text": price_linear_scale_title}]
                ),
                dict(
                    label="Log",
                    method="relayout",
                    args=[{"yaxis.type": price_log_scale_value, "yaxis.title.text": price_log_scale_title}]
                )
            ],
            font=dict(
                color="red",
                size=12,
                family="Arial"
            ),
            bgcolor="rgba(0, 0, 0, 0)",
            bordercolor="red",
            borderwidth=1
        )
    ]
)

stats = analyze_and_print_statistics(df, df_1m, discretization)
# plot_all_target_candles_multiframe_and_save_htmls_to_disc(df_trading=df, df_1m=df_1m, trading_timeframe=discretization, future_bars=500, output_dir="target_candles", clean_dir=True)
# fig_main.show()

# os.system('say "Done"')  # голосовое оповещение
# или
os.system('afplay /System/Library/Sounds/Glass.aiff')  # воспроизведение системного звука
print('\a')